In [135]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from influxdb import InfluxDBClient

client = InfluxDBClient(
    host= "10.64.44.196",
    port= 8086,
    username = "ilbekas",
    password = "!I[j~gtN25m{",
    database="workloads_IoT"
)

query = "SELECT * FROM workloads"
result = client.query(query)

points = list(result.get_points())

df = pd.DataFrame(points)

df.head()

,time,MB,bandwidth_req_mbps,data_amount_mb,dst_node,duration_s,exec_unit_id,jitter_ms,packet_loss_percent,src_node,throughput_mbps,total_energy_Wh,total_rx_Wh,total_tx_Wh
0,2025-12-23T17:36:09.117197Z,37.597839,2.4,40.0,node03,142.837769,exec_1766511369,1.865,8.1,node06,2.31,0.049112,0.010256,0.038856
1,2025-12-23T17:46:09.289206Z,137.799147,2.4,158.0,node06,570.185862,exec_1766511969,2.394,19.0,node03,1.91,0.168933,0.030614,0.138319
2,2025-12-23T17:46:11.746977Z,118.745684,2.4,138.0,node09,488.978863,exec_1766511971,1.530,19.0,node12,1.93,0.149395,0.027073,0.122322
3,2025-12-23T17:46:12.564614Z,146.549548,2.4,200.0,node10,714.699337,exec_1766511972,2.831,32.0,node11,1.61,0.172310,0.030822,0.141488
4,2025-12-23T17:54:42.095090Z,162.154786,2.4,169.0,node12,581.410376,exec_1766512482,1.439,9.8,node09,2.22,0.242146,0.040616,0.201530


In [86]:
# =========================================================
# CELL 2 — RAW DATASET STRUCTURAL INSPECTION
# =========================================================
#
# GOALS:
#
# 1. Normalize timestamps
# 2. Reconstruct workload end times
# 3. Inspect dataset structure
# 4. Inspect missing values
# 5. Inspect duplicates
# 6. Inspect temporal coverage
# 7. Inspect node/workload diversity
# 8. Generate initial numerical summaries
#
# IMPORTANT:
#
# This cell performs:
#   - NO CLEANING
#   - NO FEATURE ENGINEERING
#   - NO CONCURRENCY MODELING
#   - NO OUTLIER REMOVAL
#
# This stage is purely descriptive and analytical.
# =========================================================

import pandas as pd
import numpy as np

# =========================================================
# 1. TIMESTAMP NORMALIZATION
# =========================================================

df["time"] = pd.to_datetime(df["time"])

# =========================================================
# 2. END TIME RECONSTRUCTION
# =========================================================

df["end_time"] = (
    df["time"] +
    pd.to_timedelta(df["duration_s"], unit="s")
)

# =========================================================
# 3. DATASET SHAPE
# =========================================================

print("\n========== DATASET SHAPE ==========\n")

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

memory_mb = (
    df.memory_usage(deep=True).sum()
    / (1024 ** 2)
)

print(f"\nApproximate Memory Usage: {memory_mb:.2f} MB")

# =========================================================
# 4. COLUMN TYPES
# =========================================================

print("\n========== COLUMN TYPES ==========\n")

dtype_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str),
    "non_null_count": df.notnull().sum().values
})

print(dtype_summary)

# =========================================================
# 5. MISSING VALUE INSPECTION
# =========================================================

print("\n========== MISSING VALUES ==========\n")

missing_counts = df.isnull().sum()

missing_percent = (
    missing_counts / len(df)
) * 100

missing_df = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent
})

missing_df = missing_df[
    missing_df["missing_count"] > 0
].sort_values(
    by="missing_count",
    ascending=False
)

if len(missing_df) == 0:
    print("No missing values detected.")
else:
    print(missing_df)

# =========================================================
# 6. DUPLICATE ANALYSIS
# =========================================================

print("\n========== DUPLICATE ANALYSIS ==========\n")

duplicate_rows = df.duplicated().sum()

print(f"Duplicate full rows: {duplicate_rows}")

if "exec_unit_id" in df.columns:

    duplicate_exec_ids = (
        df["exec_unit_id"]
        .duplicated()
        .sum()
    )

    unique_exec_ids = (
        df["exec_unit_id"]
        .nunique()
    )

    print(f"Duplicate execution IDs: {duplicate_exec_ids}")
    print(f"Unique execution IDs: {unique_exec_ids}")

# =========================================================
# 7. TEMPORAL COVERAGE
# =========================================================

print("\n========== TEMPORAL COVERAGE ==========\n")

start_time = df["time"].min()
end_time = df["time"].max()

duration_days = (
    end_time - start_time
).total_seconds() / (60 * 60 * 24)

print(f"Earliest workload: {start_time}")
print(f"Latest workload:   {end_time}")

print(f"\nDataset span: {duration_days:.2f} days")

# =========================================================
# 8. TOPOLOGY & ENTITY INSPECTION
# =========================================================

print("\n========== TOPOLOGY INSPECTION ==========\n")

unique_src_nodes = (
    df["src_node"].nunique()
)

unique_dst_nodes = (
    df["dst_node"].nunique()
)

all_nodes = pd.concat([
    df["src_node"],
    df["dst_node"]
]).unique()

print(f"Unique source nodes:      {unique_src_nodes}")
print(f"Unique destination nodes: {unique_dst_nodes}")
print(f"Total unique nodes:       {len(all_nodes)}")

# unique src-dst pairs
unique_pairs = (
    df[["src_node", "dst_node"]]
    .drop_duplicates()
)

print(f"\nUnique communication pairs: {len(unique_pairs)}")

print("\nCommunication Pairs:")

print(unique_pairs.sort_values(
    by=["src_node", "dst_node"]
).to_string(index=False))

# =========================================================
# 9. NUMERICAL SUMMARY
# =========================================================

print("\n========== NUMERICAL SUMMARY ==========\n")

numeric_cols = df.select_dtypes(
    include=[
        np.number
    ]
).columns

summary = df[numeric_cols].describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99
    ]
).T

print(summary)

# =========================================================
# 10. BASIC TARGET INSPECTION
# =========================================================

print("\n========== PRIMARY TARGET OVERVIEW ==========\n")

targets = [
    "throughput_mbps",
    "duration_s",
    "total_energy_Wh",
    "packet_loss_percent",
    "jitter_ms"
]

for col in targets:

    if col not in df.columns:
        continue

    print(f"\n--- {col} ---")

    print(f"Min:   {df[col].min()}")
    print(f"Max:   {df[col].max()}")
    print(f"Mean:  {df[col].mean():.4f}")
    print(f"Std:   {df[col].std():.4f}")

# =========================================================
# 11. SAMPLE PREVIEW
# =========================================================

print("\n========== DATASET SAMPLE ==========\n")

display(df.head())


========== DATASET SHAPE ==========

Rows: 30143
Columns: 15

Approximate Memory Usage: 8.91 MB

========== COLUMN TYPES ==========

                                  column                dtype  non_null_count
time                                time  datetime64[us, UTC]           30143
MB                                    MB              float64           30143
bandwidth_req_mbps    bandwidth_req_mbps              float64           30143
data_amount_mb            data_amount_mb              float64           30143
dst_node                        dst_node                  str           30143
duration_s                    duration_s              float64           30143
exec_unit_id                exec_unit_id                  str           30143
jitter_ms                      jitter_ms              float64           30143
packet_loss_percent  packet_loss_percent              float64           30143
src_node                        src_node                  str           30143
throughp

,time,MB,bandwidth_req_mbps,data_amount_mb,dst_node,duration_s,exec_unit_id,jitter_ms,packet_loss_percent,src_node,throughput_mbps,total_energy_Wh,total_rx_Wh,total_tx_Wh,end_time
0,2025-12-23 17:36:09.117197+00:00,37.597839,2.4,40.0,node03,142.837769,exec_1766511369,1.865,8.1,node06,2.31,0.049112,0.010256,0.038856,2025-12-23 17:38:31.954966+00:00
1,2025-12-23 17:46:09.289206+00:00,137.799147,2.4,158.0,node06,570.185862,exec_1766511969,2.394,19.0,node03,1.91,0.168933,0.030614,0.138319,2025-12-23 17:55:39.475068+00:00
2,2025-12-23 17:46:11.746977+00:00,118.745684,2.4,138.0,node09,488.978863,exec_1766511971,1.530,19.0,node12,1.93,0.149395,0.027073,0.122322,2025-12-23 17:54:20.725840+00:00
3,2025-12-23 17:46:12.564614+00:00,146.549548,2.4,200.0,node10,714.699337,exec_1766511972,2.831,32.0,node11,1.61,0.172310,0.030822,0.141488,2025-12-23 17:58:07.263951+00:00
4,2025-12-23 17:54:42.095090+00:00,162.154786,2.4,169.0,node12,581.410376,exec_1766512482,1.439,9.8,node09,2.22,0.242146,0.040616,0.201530,2025-12-23 18:04:23.505466+00:00


In [87]:
# =========================================================
# CELL 3 — DATA CLEANING & DATASET SEPARATION
# =========================================================
#
# GOALS:
#
# 1. Remove physically impossible corrupted records
# 2. Define failed workload executions
# 3. Flag missing throughput rows
# 4. Reconstruct missing throughput values
# 5. Create clean modeling datasets
#
# IMPORTANT:
#
# This stage performs ONLY:
#   - integrity cleaning
#   - failure labeling
#   - dataset separation
#
# Feature engineering (concurrency, temporal, topology)
# will be performed later.
# =========================================================

# =========================================================
# CELL 3 — DATA CLEANING & DATASET SEPARATION (FIXED)
# =========================================================

import pandas as pd
import numpy as np

# =========================================================
# INITIAL STATE
# =========================================================

print("\n========== INITIAL DATASET ==========\n")
print("Rows before cleaning:", len(df))

# =========================================================
# 1. CORRUPTED VALUE DETECTION (SEPARATE FIRST)
# =========================================================

invalid_mb = (df["MB"] < 0).sum()
invalid_duration = (df["duration_s"] <= 0).sum()
invalid_energy = (df["total_energy_Wh"] < 0).sum()
invalid_throughput = (df["throughput_mbps"] > 4).sum()

print("\n========== CORRUPTED VALUE REPORT ==========\n")

print(f"Negative MB:               {invalid_mb}")
print(f"Non-positive duration:     {invalid_duration}")
print(f"Negative energy:           {invalid_energy}")
print(f"Throughput > 4 Mbps:      {invalid_throughput}")

# =========================================================
# 2. REMOVE CORRUPTED ROWS (IMPORTANT: BEFORE IMPUTATION)
# =========================================================

rows_before = len(df)

df = df[
    (df["MB"] >= 0) &
    (df["duration_s"] > 0) &
    (df["total_energy_Wh"] >= 0) &
    (df["throughput_mbps"].isna() | (df["throughput_mbps"] <= 4))
].copy()

rows_removed = rows_before - len(df)

print("\n========== CLEANING SUMMARY ==========\n")
print("Rows removed:", rows_removed)
print("Rows remaining:", len(df))

# =========================================================
# 3. FAILURE LABELING (DEFINED ON CLEANED DATA)
# =========================================================

df["execution_failed"] = (
    (df["throughput_mbps"] == 0) &
    (df["jitter_ms"] == 0) &
    (df["packet_loss_percent"] == 0)
)

print("\n========== FAILURE DISTRIBUTION ==========\n")
print(df["execution_failed"].value_counts())

print("\nFailure rate:", round(df["execution_failed"].mean() * 100, 2), "%")

# =========================================================
# 4. IMPUTED THROUGHPUT FLAG (ONLY MISSING VALUES)
# =========================================================

df["throughput_imputed"] = df["throughput_mbps"].isna()

print("\n========== MISSING THROUGHPUT ==========\n")
print("Missing values:", df["throughput_imputed"].sum())

# =========================================================
# 5. THROUGHPUT RECONSTRUCTION (ONLY WHERE NaN)
# =========================================================

missing_mask = df["throughput_mbps"].isna()

df.loc[missing_mask, "throughput_mbps"] = (
    (df.loc[missing_mask, "MB"] * 8) /
    df.loc[missing_mask, "duration_s"]
)

print("\n========== RECONSTRUCTION ==========\n")
print("Imputed rows:", missing_mask.sum())
print("Remaining NaNs:", df["throughput_mbps"].isna().sum())

# =========================================================
# 6. FINAL DATASET SPLITS (CORRECTED)
# =========================================================

# ---------------------------------------------------------
# QoS / Throughput dataset
# ---------------------------------------------------------
# Purpose:
# - throughput forecasting
# - QoS classification (jitter, loss, congestion behavior)
#
# STRICT RULES:
# - exclude failed workloads
# - exclude imputed throughput rows (synthetic targets)
# ---------------------------------------------------------

df_qos = df[
    (df["execution_failed"] == False) &
    (df["throughput_imputed"] == False)
].copy().reset_index(drop=True)


# ---------------------------------------------------------
# Energy dataset
# ---------------------------------------------------------
# Purpose:
# - total_energy_Wh prediction
# - energy-efficiency modeling
#
# RULES:
# - exclude ONLY invalid energy readings (energy == 0)
# - keep failures (they are physically meaningful for energy behavior)
# ---------------------------------------------------------

df_energy = df[
    (df["total_energy_Wh"] > 0)
].copy().reset_index(drop=True)


# ---------------------------------------------------------
# FULL dataset (for EDA + context analysis)
# ---------------------------------------------------------

df_full = df.copy().reset_index(drop=True)


# =========================================================
# 7. SANITY CHECKS
# =========================================================

print("\n========== FINAL DATASETS ==========\n")

print(f"{'Full dataset:':<30} {len(df_full)}")
print(f"{'QoS / Throughput dataset:':<30} {len(df_qos)}")
print(f"{'Energy dataset:':<30} {len(df_energy)}")


# =========================================================
# 8. FINAL THROUGHPUT STATISTICS (QoS DATASET ONLY)
# =========================================================

print("\n========== FINAL THROUGHPUT RANGE (QoS DATASET) ==========\n")

print("Min:", df_qos["throughput_mbps"].min())
print("Max:", df_qos["throughput_mbps"].max())
print("Mean:", df_qos["throughput_mbps"].mean())
print("Std:", df_qos["throughput_mbps"].std())


# =========================================================
# 9. OPTIONAL CROSS-CHECK (DATA CONSISTENCY)
# =========================================================

print("\n========== DATA INTEGRITY CHECKS ==========\n")

print("Imputed throughput excluded in QoS dataset:",
      df_qos["throughput_imputed"].sum())

print("Failed workloads in QoS dataset:",
      df_qos["execution_failed"].sum())

print("Zero energy rows in energy dataset:",
      (df_energy["total_energy_Wh"] == 0).sum())


========== INITIAL DATASET ==========

Rows before cleaning: 30143

========== CORRUPTED VALUE REPORT ==========

Negative MB:               3
Non-positive duration:     0
Negative energy:           0
Throughput > 4 Mbps:      0

========== CLEANING SUMMARY ==========

Rows removed: 3
Rows remaining: 30140

========== FAILURE DISTRIBUTION ==========

execution_failed
False    29577
True       563
Name: count, dtype: int64

Failure rate: 1.87 %

========== MISSING THROUGHPUT ==========

Missing values: 10280

========== RECONSTRUCTION ==========

Imputed rows: 10280
Remaining NaNs: 0

========== FINAL DATASETS ==========

Full dataset:                  30140
QoS / Throughput dataset:      19297
Energy dataset:                30027

========== FINAL THROUGHPUT RANGE (QoS DATASET) ==========

Min: 0.0353
Max: 2.47
Mean: 1.382890532207079
Std: 0.5814279778606004

========== DATA INTEGRITY CHECKS ==========

Imputed throughput excluded in QoS dataset: 0
Failed workloads in QoS dataset: 0
Z

In [94]:
# Cell 4 Concurrency modeling

# =========================================================
# PARAMETERS
# =========================================================
MIN_OVERLAP_SECONDS = 1.0

# =========================================================
# PREPARATION
# =========================================================
df = df.sort_values("time").reset_index(drop=True)

n = len(df)

start_times = df["time"].values
end_times = df["end_time"].values
durations = df["duration_s"].values

# =========================================================
# OUTPUT ARRAYS
# =========================================================
peak_concurrency = np.zeros(n)
overlap_seconds_total = np.zeros(n)
effective_concurrency = np.zeros(n)

# accumulate overlap ONCE per pair (i,j)
overlap_accumulator = np.zeros(n)

# =========================================================
# EVENT LIST
# =========================================================
events = []
for i in range(n):
    events.append((start_times[i], "start", i))
    events.append((end_times[i], "end", i))

events.sort(key=lambda x: x[0])

# =========================================================
# SWEEP STATE
# =========================================================
active = {}
current_active = 0

# =========================================================
# SWEEP LINE
# =========================================================
for t, typ, i in events:

    # -----------------------------------------------------
    # START EVENT
    # -----------------------------------------------------
    if typ == "start":

        start_i = start_times[i]
        end_i = end_times[i]

        instant_peak = current_active + 1
        active[i] = instant_peak

        # compare only with already-active tasks
        for j in list(active.keys()):
            if i == j:
                continue

            start_j = start_times[j]
            end_j = end_times[j]

            overlap = (
                min(end_i, end_j)
                - max(start_i, start_j)
            ).astype("timedelta64[s]").astype(float)

            if overlap >= MIN_OVERLAP_SECONDS:

                # FIX: symmetric accumulation is OK,
                # but we split attribution to avoid distortion later
                overlap_accumulator[i] += overlap
                overlap_accumulator[j] += overlap

            # update peak correctly for BOTH tasks
            active[j] = max(active[j], instant_peak)

        current_active += 1

    # -----------------------------------------------------
    # END EVENT
    # -----------------------------------------------------
    else:

        overlap_seconds_total[i] = overlap_accumulator[i]

        effective_concurrency[i] = overlap_accumulator[i] / durations[i]

        peak_concurrency[i] = active[i]

        active.pop(i, None)
        current_active -= 1

# =========================================================
# STORE RESULTS
# =========================================================
df["peak_concurrency"] = peak_concurrency
df["overlap_seconds_total"] = overlap_seconds_total
df["effective_concurrency"] = effective_concurrency

# =========================================================
# SANITY CHECKS
# =========================================================
print("\n========== CONCURRENCY FEATURES ==========\n")

print(df[[
    "peak_concurrency",
    "effective_concurrency",
    "overlap_seconds_total"
]].describe())

print("\nMax peak concurrency:", df["peak_concurrency"].max())
print("Max effective concurrency:", df["effective_concurrency"].max())


========== CONCURRENCY FEATURES ==========

       peak_concurrency  effective_concurrency  overlap_seconds_total
count      30131.000000           30131.000000           30131.000000
mean           3.939531               2.284329             415.089642
std            1.578942               1.275638             596.917613
min            1.000000               0.000000               0.000000
25%            3.000000               1.329716             158.000000
50%            4.000000               2.358476             268.000000
75%            5.000000               3.287949             443.000000
max            6.000000               4.999750           14386.000000

Max peak concurrency: 6.0
Max effective concurrency: 4.99975015248538
